# **SUPPORT**
## <span style="color:red">***Low-Power Test***</span>

### ***Initialization***

In [ ]:
"""Change to the location of the SUPPORT repo"""
path_support = "/Users/garfinkeljb/Downloads/s/SUPPORT" # NO ENDING /

In [ ]:
import sys
from pathlib import Path
import os
os.chdir(f'{path_support}/notebooks/')
sys.path.insert(0, str(Path('./').resolve().parent.parent))
import json
import torch
torch.cuda.empty_cache()
from SUPPORT.utils.train_collection import TrainingClass
from SUPPORT.utils.test_collection import TestingClass
from SUPPORT.utils.salt import simulate_lower_power
import tifffile
import numpy as np

### ***Basic configuration***
``` To modify parameters, go to notebooks/parameters.json ```

In [ ]:
dir_noisy = "/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260806_validationSet/01_data/01_raw" # input directory
dimensionality = "3+1D" # options: "2+1D" (planar + time), "3D" (static volumetric), or "3+1D" (volumetric + time)
power_percent = [100, 90, 80, 70, 60, 50, 40]

# node-dependent: change if deviating from typical node specs
n_cpu = 16
gpu = '0'

rm_tmp = False # if True, delete the noisy files after inference is complete
rm_dir = False # if True, store training artifacts under notebooks/tmp
checkpoints = False # if True, save resumable checkpoints (four times per epoch when train_gen is True)
resume = True # if True, continue from the latest complete checkpoint when available
rm_intermediaries = True # if True, keep only the most recently saved checkpoint

paths_noisy = list(Path(dir_noisy).rglob("*.tif"))
paths_noisy.sort()
for p in paths_noisy:
    print(p.parent.stem)

### ***Main loop***

In [ ]:
for i, p in enumerate(paths_noisy):
    print(f"Loading file ({i+1}/{len(paths_noisy)}): {p.stem}")
    for power in power_percent:
        exp_name = f'{p.stem}_{power:03d}'
        out_dir = f'./outputs/{p.stem}'
        pmod = Path(out_dir) / f'{exp_name}/{exp_name}.tif'

        print(f'\tSimulating {power}% power for {p.stem}')
        stack = tifffile.TiffFile(p).asarray()
        stack = simulate_lower_power(stack, power / 100, gamma=2.0, photons_per_unit=1/100)
        print(f'\tSaving low-power TIFF to {pmod}...', end=' ')
        tifffile.imwrite(pmod, stack.astype(np.float32, copy=False), imagej=True, metadata={'axes': 'TZYX'})
        print('✅')
        del stack


        path_params = "./parameters.json"
        with open(path_params, 'r') as file: params = json.load(file)
        train_params = params[dimensionality]['train']
        test_params = params[dimensionality]['test']

        train_params['dimensionality'] = dimensionality
        train_params['rm_dir'] = rm_dir
        train_params['out_dir'] = out_dir
        train_params['n_cpu'] = n_cpu
        train_params['cuda_device'] = gpu
        train_params['in_path'] = pmod
        train_params['train_gen'] = False
        train_params['exp_name'] = exp_name
        train_params['checkpoints'] = checkpoints
        train_params['resume'] = resume
        train_params['rm_intermediaries'] = rm_intermediaries
        train_dict = dict(train_params)
        train_cd = TrainingClass(train_dict)
        train_cd.vars_dict()

        train_cd.log.log_params()

        train_cd.run()
        torch.cuda.empty_cache()

        model_path = str(train_cd.model_path)
        test_in_path = train_params['in_path']
        test_params['model_path'] = model_path.replace('/vf/users', '/data')
        test_params['out_dir'] = vars(train_cd)['out_dir'].replace('/vf/users', '/data')
        test_params['exp_name'] = vars(train_cd)['exp_name']
        del train_cd

        test_dict = {
            'in_path':          test_in_path,
            'model_path':       test_params['model_path'],
            'exp_name':         test_params['exp_name'],
            'out_dir':          test_params['out_dir'],
            'dimensionality':   train_params['dimensionality'],
            'cuda_device':      train_params['cuda_device'],
            'patch_size':       train_params['patch_size'],
            'patch_interval':   test_params['patch_interval'],
            'batch_size':       test_params['batch_size'],
            'pin_memory':       test_params.get('pin_memory', True),
            'depth':            train_params['depth'],
            'bc_channels':      train_params['bc_channels'],
            'obo_channels':     train_params['obo_channels'],
            'll_channels':      train_params['ll_channels'],
            'unet_channels':    train_params['unet_channels'],
            'bs_size':          train_params['bs_size'],
            'n_cpu':            train_params['n_cpu'],
            'prefetch_factor':  test_params['prefetch_factor'],
            'pad':              test_params['pad']
        }
        test_cd = TestingClass(test_dict)

        tst = test_cd.run()
        torch.cuda.empty_cache()
        del tst, test_cd
        if rm_tmp: 
            os.remove(pmod)
